## Reconciliation Workflow

In [1]:
import pandas as pd

# Load ledger.csv
ledger_df = pd.read_csv('/content/ledger.csv')

# Load gateway.csv
gateway_df = pd.read_csv('/content/gateway.csv')

### Initial Data Overview

In [2]:
print('Ledger DataFrame Head:')
display(ledger_df.head())

print('\nGateway DataFrame Head:')
display(gateway_df.head())

Ledger DataFrame Head:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,850.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R004,2026-03-02,M003,2100.0,success,Card
4,R005,2026-03-03,M004,7200.0,success,Card



Gateway DataFrame Head:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,900.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R005,2026-03-03,M004,7200.0,failed,Card
4,R006,2026-03-03,M002,950.0,success,UPI


### Check for Duplicates and Nulls

In [3]:
print('Ledger DataFrame Info:')
ledger_df.info()
print('\nLedger DataFrame Duplicates:')
print(ledger_df.duplicated().sum())
print('\nLedger DataFrame Nulls:')
print(ledger_df.isnull().sum())

print('\n' + '='*50 + '\n')

print('Gateway DataFrame Info:')
gateway_df.info()
print('\nGateway DataFrame Duplicates:')
print(gateway_df.duplicated().sum())
print('\nGateway DataFrame Nulls:')
print(gateway_df.isnull().sum())

Ledger DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    10 non-null     object 
 1   transaction_date  10 non-null     object 
 2   merchant_id       10 non-null     object 
 3   amount_usd        10 non-null     float64
 4   status            10 non-null     object 
 5   payment_method    10 non-null     object 
dtypes: float64(1), object(5)
memory usage: 612.0+ bytes

Ledger DataFrame Duplicates:
0

Ledger DataFrame Nulls:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64


Gateway DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id  

### Identify Records Missing in Gateway

In [4]:
missing_in_gateway = ledger_df[~ledger_df['transaction_id'].isin(gateway_df['transaction_id'])]
print('Records in Ledger but missing in Gateway:')
display(missing_in_gateway)

Records in Ledger but missing in Gateway:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
3,R004,2026-03-02,M003,2100.0,success,Card
9,R010,2026-03-05,M004,2500.0,success,Wallet


### Identify Records Missing in Ledger

In [5]:
missing_in_ledger = gateway_df[~gateway_df['transaction_id'].isin(ledger_df['transaction_id'])]
print('Records in Gateway but missing in Ledger:')
display(missing_in_ledger)

Records in Gateway but missing in Ledger:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
8,R011,2026-03-05,M003,1800.0,success,Card


### Identify Amount Mismatches

In [6]:
common_transactions = pd.merge(ledger_df, gateway_df, on='transaction_id', suffixes=('_ledger', '_gateway'))

amount_mismatches = common_transactions[common_transactions['amount_usd_ledger'] != common_transactions['amount_usd_gateway']]
print('Transactions with Amount Mismatches:')
display(amount_mismatches[['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway']])

Transactions with Amount Mismatches:


,transaction_id,amount_usd_ledger,amount_usd_gateway
1,R002,850.0,900.0
6,R008,640.0,600.0


### Identify Status Mismatches

In [7]:
status_mismatches = common_transactions[common_transactions['status_ledger'] != common_transactions['status_gateway']]
print('Transactions with Status Mismatches:')
display(status_mismatches[['transaction_id', 'status_ledger', 'status_gateway']])

Transactions with Status Mismatches:


,transaction_id,status_ledger,status_gateway
3,R005,success,failed


### Final Reconciliation Report

In [8]:
# Create a combined report for missing records
report_missing_in_gateway = missing_in_gateway.copy()
report_missing_in_gateway['discrepancy_type'] = 'Missing in Gateway'

report_missing_in_ledger = missing_in_ledger.copy()
report_missing_in_ledger['discrepancy_type'] = 'Missing in Ledger'

# Create a combined report for mismatches
report_amount_mismatches = amount_mismatches[['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway']].copy()
report_amount_mismatches['discrepancy_type'] = 'Amount Mismatch'

report_status_mismatches = status_mismatches[['transaction_id', 'status_ledger', 'status_gateway']].copy()
report_status_mismatches['discrepancy_type'] = 'Status Mismatch'

# To combine, we need consistent columns. For simplicity, we'll create a list of reports and print them separately
# A more complex merge/concat would require careful column selection/renaming

print('--- Reconciliation Report ---')

print('\nRecords Missing in Gateway:')
display(report_missing_in_gateway)

print('\nRecords Missing in Ledger:')
display(report_missing_in_ledger)

print('\nTransactions with Amount Mismatches:')
display(report_amount_mismatches)

print('\nTransactions with Status Mismatches:')
display(report_status_mismatches)

--- Reconciliation Report ---

Records Missing in Gateway:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method,discrepancy_type
3,R004,2026-03-02,M003,2100.0,success,Card,Missing in Gateway
9,R010,2026-03-05,M004,2500.0,success,Wallet,Missing in Gateway



Records Missing in Ledger:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method,discrepancy_type
8,R011,2026-03-05,M003,1800.0,success,Card,Missing in Ledger



Transactions with Amount Mismatches:


,transaction_id,amount_usd_ledger,amount_usd_gateway,discrepancy_type
1,R002,850.0,900.0,Amount Mismatch
6,R008,640.0,600.0,Amount Mismatch



Transactions with Status Mismatches:


,transaction_id,status_ledger,status_gateway,discrepancy_type
3,R005,success,failed,Status Mismatch


### Summary Metrics

In [9]:
print('--- Summary Metrics ---')
print(f"Number of records missing in Gateway: {len(missing_in_gateway)}")
print(f"Number of records missing in Ledger: {len(missing_in_ledger)}")
print(f"Number of transactions with amount mismatches: {len(amount_mismatches)}")
print(f"Number of transactions with status mismatches: {len(status_mismatches)}")

--- Summary Metrics ---
Number of records missing in Gateway: 2
Number of records missing in Ledger: 1
Number of transactions with amount mismatches: 2
Number of transactions with status mismatches: 1


## Part 4: JSON Normalization

### Read and Flatten Nested JSON File

In [16]:
import json
from pandas import json_normalize

# Load the JSON file
with open('/content/api_response_sample.json', 'r') as f:
    api_data = json.load(f)

# First flatten the 'batches' list and keep 'generated_at' and 'source' as metadata.
df_batches = json_normalize(
    api_data,
    record_path=['batches'],
    meta=['generated_at', 'source'],
    sep='_'
)

# Now, further flatten the 'settlements' list within each batch
# We need to extract the 'settlements' data along with the 'batch_id' and other relevant batch info

# Prepare data for second level normalization
list_of_settlements = []
for index, row in df_batches.iterrows():
    batch_id = row['batch_id']
    generated_at = row['generated_at']
    source = row['source']
    merchant_id = row['merchant_merchant_id']
    merchant_name = row['merchant_merchant_name']
    merchant_region = row['merchant_region']

    if 'settlements' in row and isinstance(row['settlements'], list):
        for settlement in row['settlements']:
            settlement_record = settlement.copy()
            settlement_record['batch_id'] = batch_id
            settlement_record['generated_at'] = generated_at
            settlement_record['source'] = source
            settlement_record['merchant_merchant_id'] = merchant_id
            settlement_record['merchant_merchant_name'] = merchant_name
            settlement_record['merchant_region'] = merchant_region
            list_of_settlements.append(settlement_record)

df_normalized = pd.DataFrame(list_of_settlements)

# Flatten the 'bank' column
df_normalized['bank_name'] = df_normalized['bank'].apply(lambda x: x.get('name'))
df_normalized['bank_country'] = df_normalized['bank'].apply(lambda x: x.get('country'))
df_normalized = df_normalized.drop(columns=['bank'])

print('Normalized DataFrame Head:')
display(df_normalized.head())

print('\nNormalized DataFrame Info:')
df_normalized.info()

Normalized DataFrame Head:


,settlement_id,amount_usd,status,processed_at,batch_id,generated_at,source,merchant_merchant_id,merchant_merchant_name,merchant_region,bank_name,bank_country
0,S001,1520.5,settled,2026-03-07T08:10:00Z,B001,2026-03-07T10:00:00Z,QuickPay Settlement API,M001,Alpha Mart,APAC,Bank A,IN
1,S002,980.0,pending,2026-03-07T08:45:00Z,B001,2026-03-07T10:00:00Z,QuickPay Settlement API,M001,Alpha Mart,APAC,Bank A,IN
2,S003,640.0,settled,2026-03-07T09:15:00Z,B001,2026-03-07T10:00:00Z,QuickPay Settlement API,M001,Alpha Mart,APAC,Bank B,SG
3,S004,2100.0,settled,2026-03-07T08:20:00Z,B002,2026-03-07T10:00:00Z,QuickPay Settlement API,M004,Delta Travels,US,Bank C,US
4,S005,500.0,failed,2026-03-07T08:50:00Z,B002,2026-03-07T10:00:00Z,QuickPay Settlement API,M004,Delta Travels,US,Bank C,US



Normalized DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   settlement_id           6 non-null      object 
 1   amount_usd              6 non-null      float64
 2   status                  6 non-null      object 
 3   processed_at            6 non-null      object 
 4   batch_id                6 non-null      object 
 5   generated_at            6 non-null      object 
 6   source                  6 non-null      object 
 7   merchant_merchant_id    6 non-null      object 
 8   merchant_merchant_name  6 non-null      object 
 9   merchant_region         6 non-null      object 
 10  bank_name               6 non-null      object 
 11  bank_country            6 non-null      object 
dtypes: float64(1), object(11)
memory usage: 708.0+ bytes


### Clean Column Names

In [11]:
import re

def clean_column_name(col_name):
    col_name = col_name.lower() # Convert to lowercase
    col_name = re.sub(r'[^a-z0-9_]', '_', col_name) # Replace non-alphanumeric with underscore
    col_name = re.sub(r'__+', '_', col_name) # Replace multiple underscores with single
    col_name = col_name.strip('_') # Remove leading/trailing underscores
    return col_name

df_normalized.columns = [clean_column_name(col) for col in df_normalized.columns]

print('Cleaned Column Names:')
print(df_normalized.columns.tolist())

Cleaned Column Names:
['generated_at', 'source', 'batches']


### Convert Date/Time Fields

In [12]:
# Identify potential date/time columns based on common naming conventions
datetime_cols = [
    col for col in df_normalized.columns if 'date' in col or 'timestamp' in col or 'at' in col
]

for col in datetime_cols:
    try:
        # Attempt to convert to datetime, coercing errors to NaT
        df_normalized[col] = pd.to_datetime(df_normalized[col], errors='coerce')
    except Exception as e:
        print(f"Could not convert column '{col}' to datetime: {e}")

print('\nDataFrame Info after datetime conversion:')
df_normalized.info()


DataFrame Info after datetime conversion:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   generated_at  1 non-null      datetime64[ns, UTC]
 1   source        1 non-null      object             
 2   batches       0 non-null      datetime64[ns]     
dtypes: datetime64[ns, UTC](1), datetime64[ns](1), object(1)
memory usage: 156.0+ bytes


### Save Normalized Output

In [13]:
# Save the normalized DataFrame to a new CSV file
output_file_path = '/content/normalized_api_data.csv'
df_normalized.to_csv(output_file_path, index=False)

print(f"Normalized data saved to {output_file_path}")

Normalized data saved to /content/normalized_api_data.csv
